# 04 - Final Submission
**Kompetisi:** INFEST XII 2026 - Data Science (Klasifikasi Aksara Tradisional Nusantara)
**Tim:** Fadhil - Percobaan 1

Notebook ini adalah tahap akhir: load model terbaik (`best_model.pt`, Val F1 Macro 0.9939),
jalankan prediksi pada `test.csv`, dan generate `submission.csv` sesuai format yang diminta panitia.

**Ringkasan hasil dari notebook sebelumnya:**
- Model: ResNet18 pretrained + fine-tuning
- Val F1 Macro: 0.9939
- Kelas paling menantang: `jawi` dan `pegon` (sama-sama basis aksara Arab, saling tertukar di
  beberapa kasus ambigu)

**Isi notebook:**
1. Setup & Load Model Terbaik
2. Load Test Set
3. Prediksi pada Test Set
4. Validasi Format Submission
5. Generate & Simpan submission.csv
6. Ringkasan Akhir (untuk Notebook Gabungan / Laporan)


## 1. Setup & Load Model Terbaik

Set random seed untuk reproduksibilitas penuh, sesuai ketentuan lomba bahwa notebook harus bisa
dijalankan ulang (di-run) tanpa error dan menghasilkan output yang konsisten.

In [ ]:
import os
import json
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
from PIL import Image

# Reproducibility — WAJIB dijalankan di awal
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

DATA_DIR = "../dataset"
TEST_CSV = os.path.join(DATA_DIR, "test.csv")
TEST_IMG_DIR = os.path.join(DATA_DIR, "images", "test")
SAMPLE_SUB_CSV = os.path.join(DATA_DIR, "sample_submission.csv")
ARTIFACT_DIR = "../artifacts"
OUTPUT_DIR = "../submissions"
os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(os.path.join(ARTIFACT_DIR, "preprocessing_config.json")) as f:
    config = json.load(f)

TARGET_SIZE = tuple(config["target_size"])
classes_sorted = config["classes"]
label_to_idx = config["label_to_idx"]
idx_to_label = {v: k for k, v in label_to_idx.items()}

print("Classes:", classes_sorted)
print("Target size:", TARGET_SIZE)


In [ ]:
def resize_with_padding(img, target_size=TARGET_SIZE, fill_color=(255, 255, 255)):
    img = img.convert("RGB")
    original_w, original_h = img.size
    target_w, target_h = target_size

    ratio = min(target_w / original_w, target_h / original_h)
    new_w, new_h = int(original_w * ratio), int(original_h * ratio)

    img_resized = img.resize((new_w, new_h), Image.LANCZOS)

    new_img = Image.new("RGB", target_size, fill_color)
    paste_x = (target_w - new_w) // 2
    paste_y = (target_h - new_h) // 2
    new_img.paste(img_resized, (paste_x, paste_y))

    return new_img

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
to_tensor_normalize = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


def build_resnet_model(num_classes):
    model = models.resnet18(weights=None)
    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, num_classes)
    return model


checkpoint = torch.load(os.path.join(ARTIFACT_DIR, "best_model.pt"), map_location=DEVICE)

model = build_resnet_model(num_classes=len(classes_sorted)).to(DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print(f"Model checkpoint di-load dari epoch {checkpoint['epoch']+1}, "
      f"Val F1 Macro tercatat: {checkpoint['val_f1']:.4f}")

assert checkpoint["classes"] == classes_sorted, \
    "Urutan kelas checkpoint tidak sama dengan config saat ini — cek konsistensi sebelum lanjut!"


## 2. Load Test Set

In [ ]:
test_df = pd.read_csv(TEST_CSV)
sample_sub_df = pd.read_csv(SAMPLE_SUB_CSV)

print("Test set shape:", test_df.shape)
print("Sample submission shape:", sample_sub_df.shape)
test_df.head()


In [ ]:
# Sanity check: pastikan semua image_id di test.csv punya file gambarnya
test_files = set(os.listdir(TEST_IMG_DIR))
missing_test_imgs = set(test_df['image_id']) - test_files
print(f"image_id di test.csv tapi file tidak ditemukan: {len(missing_test_imgs)}")
assert len(missing_test_imgs) == 0, "Ada file test yang hilang! Cek ulang folder images/test."


## 3. Prediksi pada Test Set

Menggunakan pipeline preprocessing **yang sama persis** dengan validation set (resize+padding,
normalisasi ImageNet), **tanpa augmentasi** — karena augmentasi hanya untuk training.

In [ ]:
class AksaraTestDataset(Dataset):
    def __init__(self, df, img_dir):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row['image_id'])
        img = Image.open(img_path)
        img = resize_with_padding(img)
        tensor = to_tensor_normalize(img)
        return tensor, row['image_id']


test_dataset = AksaraTestDataset(test_df, TEST_IMG_DIR)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

print(f"Test batches: {len(test_loader)}")


In [ ]:
@torch.no_grad()
def predict_test(model, loader, device):
    all_preds, all_ids, all_probs = [], [], []

    for images, img_ids in loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        preds = outputs.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_ids.extend(img_ids)
        all_probs.extend(probs.cpu().numpy())

    return np.array(all_preds), list(all_ids), np.array(all_probs)


test_preds, test_ids, test_probs = predict_test(model, test_loader, DEVICE)

submission_df = pd.DataFrame({
    "image_id": test_ids,
    "label": [idx_to_label[p] for p in test_preds],
})

print(f"Total prediksi: {len(submission_df)}")
submission_df.head()


In [ ]:
# Cek distribusi prediksi -- bandingkan dengan distribusi kelas di train set
# untuk memastikan model tidak collapse ke 1-2 kelas saja
pred_distribution = submission_df['label'].value_counts().sort_values(ascending=False)
print("Distribusi prediksi pada test set:")
print(pred_distribution)
print()
print(f"Confidence rata-rata: {test_probs.max(axis=1).mean():.4f}")
print(f"Confidence minimum  : {test_probs.max(axis=1).min():.4f}")


In [ ]:
# Cek sampel dengan confidence RENDAH -- kandidat prediksi yang paling tidak pasti,
# berguna untuk laporan/diskusi, bukan untuk diubah manual
low_conf_mask = test_probs.max(axis=1) < 0.7
low_conf_df = submission_df[low_conf_mask].copy()
low_conf_df["confidence"] = test_probs.max(axis=1)[low_conf_mask]
print(f"Jumlah prediksi dengan confidence < 0.7: {len(low_conf_df)} dari {len(submission_df)}")
low_conf_df.sort_values("confidence").head(10)


## 4. Validasi Format Submission

Sesuai guidebook: file submission wajib berkolom `image_id,label`, dan jumlah baris harus sama
dengan jumlah gambar pada test set. Validasi ini WAJIB dijalankan sebelum submit ke Kaggle.

In [ ]:
# 1. Cek kolom
assert list(submission_df.columns) == ["image_id", "label"], \
    f"Kolom tidak sesuai format! Ditemukan: {list(submission_df.columns)}"

# 2. Cek jumlah baris sama dengan test set
assert len(submission_df) == len(test_df), \
    f"Jumlah baris submission ({len(submission_df)}) != jumlah test set ({len(test_df)})"

# 3. Cek semua image_id di submission sama persis dengan yang ada di test.csv (tidak kurang/lebih)
assert set(submission_df["image_id"]) == set(test_df["image_id"]), \
    "Ada image_id yang hilang atau berlebih dibanding test.csv!"

# 4. Cek tidak ada nilai kosong
assert submission_df.isnull().sum().sum() == 0, "Ada nilai kosong (NaN) di submission!"

# 5. Cek semua label yang diprediksi valid (termasuk dalam 7 kelas yang diketahui)
invalid_labels = set(submission_df["label"]) - set(classes_sorted)
assert len(invalid_labels) == 0, f"Ditemukan label tidak valid: {invalid_labels}"

# 6. Cek tidak ada duplikat image_id
assert submission_df["image_id"].duplicated().sum() == 0, "Ada image_id duplikat di submission!"

print("Semua validasi format LOLOS. Submission siap disimpan.")


## 5. Generate & Simpan submission.csv

Urutan baris disamakan dengan `sample_submission.csv` / `test.csv` agar konsisten dengan
ekspektasi sistem submission Kaggle.

In [ ]:
# Urutkan submission_df sesuai urutan image_id pada test.csv (bukan urutan hasil inferensi)
submission_df = submission_df.set_index("image_id").loc[test_df["image_id"]].reset_index()

submission_path = os.path.join(OUTPUT_DIR, "submission.csv")
submission_df.to_csv(submission_path, index=False)

print(f"submission.csv berhasil disimpan di: {submission_path}")
print(f"Jumlah baris: {len(submission_df)}")
submission_df.head()


In [ ]:
# Verifikasi ulang file yang baru disimpan (baca ulang dari disk, bukan dari memory)
check_df = pd.read_csv(submission_path)
assert check_df.shape == (len(test_df), 2), "Shape file tersimpan tidak sesuai ekspektasi!"
assert list(check_df.columns) == ["image_id", "label"], "Kolom file tersimpan tidak sesuai!"
print("Verifikasi ulang file tersimpan: OK, siap diunggah ke Kaggle.")


## 6. Ringkasan Akhir (untuk Notebook Gabungan / Laporan)

**Ringkasan pipeline end-to-end (Percobaan 1):**

| Tahap | Keputusan | Alasan Singkat |
|---|---|---|
| Cleaning | Drop 1 file corrupt | File tidak bisa dibuka, <0.1% data |
| Split | Stratified 85/15 | Jaga proporsi 7 kelas (imbalance 2.24x) |
| Resize | Letterbox (resize + padding) | Aspect ratio gambar sangat bervariasi |
| Augmentasi | Ringan (mayoritas) / agresif (bali, lontara) | Atasi imbalance tanpa merusak bentuk aksara |
| Model | ResNet18 pretrained, full fine-tuning | Dataset kecil, transfer learning membantu |
| Loss | CrossEntropyLoss + class weight | Selaras dengan metrik F1 Macro |
| Val F1 Macro | **0.9939** | Error terkonsentrasi di pasangan `jawi`-`pegon` |

**Temuan kunci untuk laporan/final presentation:**
- Dataset relatif bersih (0 duplikat, hanya 1 file corrupt), tapi ditemukan indikasi mislabel
  minor pada kelas `jawi` (beberapa sampel berisi teks Latin, bukan aksara Jawi)
- 100% error model (5 dari 671 sampel val) terjadi antara `jawi` dan `pegon` — dua aksara yang
  sama-sama berbasis huruf Arab dengan gaya tulisan tangan yang mirip secara visual
- Model tidak collapse ke kelas tertentu — distribusi prediksi pada test set perlu dibandingkan
  dengan distribusi label training (lihat Section 3) untuk memastikan konsistensi

**Reproduksibilitas:**
- Semua random seed (`numpy`, `torch`, `cuda`) di-set di awal setiap notebook
- Path dataset menggunakan path relatif, bukan absolute path
- Artefak intermediate (split data, config, model checkpoint) disimpan di folder `artifacts/`
  agar tiap notebook dapat dijalankan ulang secara independen dan konsisten

**File yang dihasilkan:**
- `../submissions/submission.csv` — siap diunggah ke Kaggle
- `../artifacts/best_model.pt` — model checkpoint terbaik
- `../artifacts/training_history.json` — log training untuk dokumentasi

**Next steps (jika ada waktu untuk iterasi lanjutan / percobaan-2):**
1. Investigasi & bersihkan sampel mislabel di kelas `jawi` pada training set
2. Coba arsitektur lebih besar (ResNet34/EfficientNet) untuk melihat apakah pasangan `jawi`-`pegon`
   bisa dipisahkan lebih baik lagi
3. Cross-validation (K-Fold) untuk memastikan skor 0.99 stabil, bukan kebetulan dari 1 split saja